# reachscan — run on any open-weights model

This notebook installs reachscan from the repository and runs a reach-scan on an open-weights model: it measures how target reachability changes as a committed reasoning prefix deepens. Running it on a model the instrument was not built around is a cross-model check, not a reproduction of the paper.

## What this can run on
- Any open-weights, autoregressive model you can load and sample from at the token level: a HuggingFace model (the typical path), a local model, or a frontier model if you hold its weights. The requirement is token-level access, not the public/closed distinction.
- Not a hosted chat API (e.g. Claude/GPT endpoints): the measurement freezes a committed prefix on token IDs and samples fresh continuations, which an API does not expose. Non-autoregressive substrates (e.g. diffusion) are a research extension of the abstract contract and are not implemented here.

## Requirements
- GPU runtime (Runtime → Change runtime type → GPU). An L4 or A100 is comfortable for an 8B model; a T4 suffices for the smoke test.
- Gated models (e.g. Llama-3.1) require accepting the license on the model's HuggingFace page and authenticating with a token. Without access, use the ungated smoke model below to verify the pipeline.

## 1 · Install reachscan from your repo

In [ ]:
# Point this at your repo once reachscan is pushed. Until then, change to your branch/path.
REPO_URL = "git+https://github.com/ReachabilityLabs/ReachScan.git"   # <-- adjust to your real repo/branch

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"reachscan[hf,plot] @ {REPO_URL}"])

# Fallback if not yet public — upload the zip to Colab and:
# Offline fallback: upload reachscan_v0_2_PUBLIC.zip to /content, then:
# !cd /content && unzip -q reachscan_v0_2_PUBLIC.zip && pip install -q "./reachscan[hf,plot]"
import reachscan; print('reachscan', reachscan.__version__, 'installed')

## 2 · Pick a model
Start with the **smoke model** to confirm `hf_source.py` actually runs on weights (the one thing the test suite can't check). Then set `MODEL_ID` to **your** target open-weights model for the real run — `meta-llama/Llama-3.1-8B-Instruct` is shown as one example, but any open-weights autoregressive HF model (or local path) works.

In [ ]:
# SMOKE (ungated, fast — verifies hf_source.py runs on real weights):
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# REAL RUN — set this to ANY open-weights autoregressive model you can load
# (a HuggingFace id or a local path). Example (gated, accept the license first):
# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

REVISION = None   # pin to a commit hash for reproducibility

# Gated models need a token — accept the license on HF, then log in:
# from huggingface_hub import login; login()   # paste your HF token

from reachscan.hf_source import HuggingFaceSource
source = HuggingFaceSource(MODEL_ID, revision=REVISION)
print('loaded', source.name)

## 3 · Define the problem and the projection
`ExactMatch(ground_truth)` measures reachability of the **exact** correct answer — the cleanest target-relative signal. Swap in `ModuloProjection(8, target_residue=...)` to see family-level foreclosure instead. Set the problem and its known correct answer.

In [ ]:
from reachscan import ExactMatch, ModuloProjection, SamplerPolicy, GeneratedPrefixSource

PROMPT = (
    "Please reason step by step, and put your final answer within \\boxed{}. "
    "Problem: Compute the sum of floor((3n+7)/5) for n = 1 through n = 40."
)
GROUND_TRUTH = 532          # <-- the known correct answer for THIS problem

projection = ExactMatch(GROUND_TRUTH)
# projection = ModuloProjection(8, target_residue=GROUND_TRUTH % 8)  # family view
print('projection:', projection.name)

## 3b · Does the extractor catch THIS model's answer format?
The shipped extractor expects a `\\boxed{}`-style answer. A **general** model (not math-tuned) may not emit that reliably — if it doesn't, answer yield drops and R_T gets noisy. Inspect a few raw completions **before** trusting any number.

In [ ]:
committed = source.encode_prompt(PROMPT)   # prompt-only state (f = 0)
ok = 0
for r in range(5):
    ids = source.sample_completion(committed, temperature=0.7, top_p=1.0,
                                   max_new_tokens=512, seed=1000 + r)
    ans = projection.extract(source.decode(ids))
    ok += ans.is_ok
    print(f"[{ans.status:9}] extracted={ans.value!r}")
print(f"\nyield: {ok}/5 ok. If most are 'ok' with sane numbers, the extractor fits this model.")
print("If many are 'no_answer', tighten PROMPT (force \\boxed{}) or supply a custom extract before scanning.")


## 4 · Smoke run — verify the real-model path end to end
A tiny budget at two depths. If it prints a table with sane numbers, the HuggingFace source runs correctly on weights. Run the full scan only after this passes.

In [ ]:
from reachscan import reach_scan, DepthSpec, uniform_plan

smoke_prefix = GeneratedPrefixSource(
    source, PROMPT, trace_sampler=SamplerPolicy(max_new_tokens=512), seed=0
)
smoke = reach_scan(
    source=source, prefix_source=smoke_prefix, projection=projection,
    plan=uniform_plan([0.0, 1.0], 8),
    rollout_sampler=SamplerPolicy(max_new_tokens=512), base_seed=0,
)
for s in smoke.summaries:
    print(f'f={s.fraction:.2f}  ok={s.ok_answers:>2}  R_T={s.target_reachability:.3f}  dom={s.dominant_bucket}')
print('\nSmoke OK — the file runs on weights.' )

## 4b · Declare the full-run plan and inspect resources

Before the full scan, declare the rollout plan and print the resource facts that determine wall-clock risk: GPU, memory, model placement, trace budget, rollout budget, and total generations. These diagnostics are provenance for deciding whether a run is appropriately sized for the runtime.

In [ ]:
from pathlib import Path
import json, platform, time

RUN_DIR = Path('reachscan_run')
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
TRACE_CACHE = RUN_DIR / 'frozen_trace.json'

plan = [
    DepthSpec(0.00, 128),
    DepthSpec(0.25,  64),
    DepthSpec(0.50,  64),
    DepthSpec(0.75, 128),
    DepthSpec(0.90, 128),
    DepthSpec(1.00, 128),
]
trace_sampler = SamplerPolicy(max_new_tokens=2048)
rollout_sampler = SamplerPolicy(max_new_tokens=512)

print('source:', source.name)
print('python:', platform.python_version())
print('plan:', [(d.fraction, d.rollouts) for d in plan])
print('total rollout generations:', sum(d.rollouts for d in plan))
print('trace max_new_tokens:', trace_sampler.max_new_tokens)
print('rollout max_new_tokens:', rollout_sampler.max_new_tokens)

try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print('gpu memory GB:', round(props.total_memory / 1024**3, 2))
        print('allocated GB:', round(torch.cuda.memory_allocated(0) / 1024**3, 2))
        print('reserved GB:', round(torch.cuda.memory_reserved(0) / 1024**3, 2))
except Exception as exc:
    print('torch diagnostics unavailable:', repr(exc))

print('model device:', getattr(getattr(source, '_model', None), 'device', None))
print('hf device map:', getattr(getattr(source, '_model', None), 'hf_device_map', None))
print('checkpoint dir:', CHECKPOINT_DIR)


## 5 · Full reach-scan with per-depth checkpoints

Generate or reload one frozen reference trace, then run one depth at a time. Each completed depth is written under `reachscan_run/checkpoints/depth_XXX/`. If Colab disconnects, rerun the notebook: completed depths are loaded from disk and skipped, while missing depths keep their original depth indices and seeds.

In [ ]:
from reachscan import ReachScanResult, UserPrefixSource
from reachscan.metadata import read_result, write_result

RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if TRACE_CACHE.exists():
    trace_doc = json.loads(TRACE_CACHE.read_text(encoding='utf-8'))
    prefix_source = UserPrefixSource(
        trace_doc['prompt_ids'],
        trace_doc['reference_trace_ids'],
        name=trace_doc.get('name', 'checkpointed_trace'),
    )
    prefix_source.provenance = trace_doc.get('provenance', {'kind': 'checkpointed_trace'})
    print('loaded frozen trace from', TRACE_CACHE)
else:
    prefix_source = GeneratedPrefixSource(
        source, PROMPT, trace_sampler=trace_sampler, seed=0
    )
    trace_doc = {
        'name': getattr(prefix_source, 'name', 'generated_trace'),
        'prompt_ids': prefix_source.prompt_ids(),
        'reference_trace_ids': prefix_source.reference_trace_ids(),
        'provenance': getattr(prefix_source, 'provenance', None),
    }
    TRACE_CACHE.write_text(json.dumps(trace_doc, indent=2), encoding='utf-8')
    print('generated and cached frozen trace at', TRACE_CACHE)

trace_len = len(prefix_source.reference_trace_ids())
print('frozen trace tokens:', trace_len)
if trace_len >= trace_sampler.max_new_tokens:
    print('warning: reference trace hit the trace token cap; treat deep fractions carefully')

expected_plan = [
    {'fraction': d.fraction, 'rollouts': d.rollouts, 'committed_len': d.committed_len}
    for d in plan
]

def validate_checkpoint(part, depth_index, checkpoint_path):
    manifest = part.manifest
    idxs = manifest.get('executed_depth_indices') or sorted({r.depth_index for r in part.receipts})
    if idxs != [depth_index]:
        raise ValueError(f'checkpoint {checkpoint_path} has depth indices {idxs}, expected [{depth_index}]')
    if manifest.get('source') != source.name:
        raise ValueError(f'checkpoint {checkpoint_path} source differs from current source')
    if manifest.get('projection') != projection.name:
        raise ValueError(f'checkpoint {checkpoint_path} projection differs from current projection')
    if manifest.get('trace_len') != trace_len:
        raise ValueError(f'checkpoint {checkpoint_path} trace length differs from frozen trace')
    if manifest.get('base_seed') != 0:
        raise ValueError(f'checkpoint {checkpoint_path} base_seed differs from current run')
    if manifest.get('rollout_sampler') != vars(rollout_sampler):
        raise ValueError(f'checkpoint {checkpoint_path} rollout sampler differs from current run')
    checkpoint_plan = manifest.get('plan', [])
    if len(checkpoint_plan) != len(expected_plan):
        raise ValueError(f'checkpoint {checkpoint_path} plan length differs from current plan')
    got = checkpoint_plan[depth_index]
    expected = expected_plan[depth_index]
    if got.get('fraction') != expected['fraction'] or got.get('rollouts') != expected['rollouts'] or got.get('committed_len') != expected['committed_len']:
        raise ValueError(f'checkpoint {checkpoint_path} plan row differs from current plan')

parts_by_depth = {}
for depth_index in range(len(plan)):
    checkpoint_path = CHECKPOINT_DIR / f'depth_{depth_index:03d}'
    if (checkpoint_path / 'summary_by_depth.csv').exists() and (checkpoint_path / 'receipts.csv').exists():
        part = read_result(checkpoint_path)
        validate_checkpoint(part, depth_index, checkpoint_path)
        parts_by_depth[depth_index] = part
        print(f'loaded checkpoint depth_index={depth_index}: {checkpoint_path}')

for depth_index, spec in enumerate(plan):
    if depth_index in parts_by_depth:
        continue
    checkpoint_path = CHECKPOINT_DIR / f'depth_{depth_index:03d}'
    print(f'running depth_index={depth_index} f={spec.fraction:.2f} M={spec.rollouts}')
    started = time.time()
    part = reach_scan(
        source=source,
        prefix_source=prefix_source,
        projection=projection,
        plan=plan,
        rollout_sampler=rollout_sampler,
        base_seed=0,
        run_depth_indices=[depth_index],
    )
    write_result(part, checkpoint_path)
    parts_by_depth[depth_index] = part
    elapsed = time.time() - started
    s = part.summaries[0]
    print(
        f'checkpointed depth_index={depth_index} in {elapsed/60:.1f} min: '
        f'ok={s.ok_answers}/{s.attempts}, R_T={s.target_reachability:.3f}, '
        f'cap_hits={s.cap_hits}'
    )

missing = [i for i in range(len(plan)) if i not in parts_by_depth]
if missing:
    raise RuntimeError(f'missing checkpoint depths after run: {missing}')

result = ReachScanResult()
for depth_index in range(len(plan)):
    part = parts_by_depth[depth_index]
    result.summaries.extend(part.summaries)
    result.receipts.extend(part.receipts)

first_manifest = parts_by_depth[0].manifest.copy()
first_manifest.pop('executed_depth_indices', None)
result.manifest = first_manifest
out = write_result(result, RUN_DIR)

print(f"{'depth':>6} {'M':>5} {'ok':>7} {'R_T':>6} {'dom':>6} {'dom_mass':>8} {'entropy':>7} {'cap':>5}")
for s in result.summaries:
    print(f'{s.fraction:>6.2f} {s.attempts:>5} {s.ok_answers:>7} {s.target_reachability:>6.3f} '
          f'{str(s.dominant_bucket):>6} {s.dominant_mass:>8.3f} {s.answer_field_entropy:>7.3f} {s.cap_hits:>5}')
print('final artifacts written to', out)


## 6 · Inspect saved artifacts

The final artifacts live in `reachscan_run/`. Per-depth checkpoint artifacts live under `reachscan_run/checkpoints/`. The final files are the ones to copy into the repository example after the run is reviewed.

In [ ]:
from pathlib import Path
for p in sorted(Path('reachscan_run').glob('*')):
    print(p)
print('\nfinal files:')
for p in sorted(Path('reachscan_run').iterdir()):
    print(f'{p.stat().st_size:>10} bytes  {p.name}')


## 7 · Plot the gap

In [ ]:
import matplotlib.pyplot as plt
f   = [s.fraction for s in result.summaries]
rt  = [s.target_reachability for s in result.summaries]
ent = [s.answer_field_entropy for s in result.summaries]
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(f, rt, 'o-', color='#B87333', label='target reachability R_T(f)')
ax.set_xlabel('committed-prefix depth'); ax.set_ylabel('R_T(f)'); ax.set_ylim(0,1)
ax2 = ax.twinx(); ax2.plot(f, ent, 's--', color='#0D9B8A', alpha=.6, label='answer-field entropy')
ax2.set_ylabel('entropy (bits)')
ax.set_title(f'reach-scan: {source.name}'); ax.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.tight_layout(); plt.show()

## 8 · How to read it
- **Check the `ok` (answer-yield) column first.** Low yield means the model is not emitting parseable answers (a format/extractor issue), not low reachability. Also check cap-hits (generations cut at `max_new_tokens`).
- **A sustained fall in R_T(f) before answer exposure** indicates foreclosure under the declared sampler — the committed prefix is losing access to the correct answer. A single noisy run can wiggle, so look for a sustained trend, not one dip.
- **Late depths, especially f=1.0,** may already contain the answer in the committed trace; treat those as post-hoc morphology unless answer exposure is checked.
- **Entropy falling while R_T falls** indicates the field concentrating into a wrong basin (capture, not diffusion).
- **A dominant bucket that is not the target taking over** is where the wrong family wins.

This is one model, one problem, one sampler — a measurement, not a law. Its value is a second substrate: if the same shape appears on a model the instrument was not tuned on, that is evidence the effect is not specific to the original Qwen floor-sum case. The saved artifacts are an inspectable example output.